In [1]:
import pandas as pd, os, datetime
import numpy as np
from scipy import stats
from sklearn.cluster import KMeans
from sklearn import metrics
import numpy as np
from sklearn.metrics import silhouette_samples, silhouette_score
import geopandas as gpd
import warnings

# For plotting
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import seaborn as sns
import cartopy.io.img_tiles as cimgt
pio.renderers.default = 'notebook'

# Import the loader function from your module
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
from process_code import load_generation_data

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning:

The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/distributed/node.py:187: UserWarning:

Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41855 instead

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-

Dask dashboard: /proxy/41855/status


/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that i

In [2]:
df, info = load_generation_data(
    sdate="2009-07-01",
    edate="2024-06-30",
    mode="daily",
    ftype=["Wind"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=True,
    wind_zero_threshold=40,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

Read gen_details & hw_tseries with Dask: 0.05 sec
Select group: 0.00 sec
--- Starting Dask-Native Process ---


/g/data/ng72/ms5578/ID_HW_BARRA/process_code.py:168: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Starting final Dask compute...


/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/dask/dataframe/io/csv.py:77: DtypeWarning: Columns (2,3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/dask/dataframe/io/csv.py:77: DtypeWarning: Columns (2,3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/dask/dataframe/io/csv.py:77: DtypeWarning: Columns (2,3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)
2026-01-16 09:47:29,922 - distributed.worker - ERROR - Compute Failed
Key:       ('read-csv-55bd83a40da7ae65ca911aeb9620c8cc', 481)
State:     executing
Task:  <Task ('read-csv-55bd83a40da7ae65ca911aeb9620c8cc', 481) _read_csv(..., ...)>
Exception: 'ValueErro

ValueError: Mismatched dtypes found in `pd.read_csv`/`pd.read_table`.

+--------------+--------+----------+
| Column       | Found  | Expected |
+--------------+--------+----------+
| AGCSTATUS    | object | float64  |
| TOTALCLEARED | object | float64  |
| TOTALMWh     | object | float64  |
+--------------+--------+----------+

The following columns also raised exceptions on conversion:

- AGCSTATUS
  ValueError("could not convert string to float: 'AGCSTATUS'")
- TOTALCLEARED
  ValueError("could not convert string to float: 'TOTALCLEARED'")
- TOTALMWh
  ValueError("could not convert string to float: 'TOTALMWh'")

Usually this is due to dask's dtype inference failing, and
*may* be fixed by specifying dtypes manually by adding:

dtype={'AGCSTATUS': 'object',
       'TOTALCLEARED': 'object',
       'TOTALMWh': 'object'}

to the call to `read_csv`/`read_table`.

In [ ]:
info = info[info['DUID'].isin(df['DUID'])]

In [ ]:
df = df.copy()
df['time'] = pd.to_datetime(df['time'])
df['date'] = df['time'].dt.date

# Count unique heatwave days per DUID
heatwave_days = (
    df[df['EHF_flag'] == 1]
    .groupby('DUID')['date']
    .nunique()
)
heatwave_days

This doesn't work because I removed the jittering

In [ ]:
# Plot all wind farms
fig = px.scatter_map(
    info,
    lat='lat_jittered',
    lon='lon_jittered',
    hover_name='DUID',
    hover_data={
        'lat': False,
        'lon': False,
        'lat_jittered': False,
        'lon_jittered': False,
        'fuel_source_primary': True,
    },
    color='fuel_source_primary',
    
    zoom=3,
    map_style='carto-darkmatter'
)
fig.update_traces(marker=dict(size=5))

fig.update_layout(
    legend=dict(
        title="Fuel Type",
    ),
    title_text="Locations of Generation Units"
)

fig.show()

In [ ]:
#Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(info, geometry=gpd.points_from_xy(info.lon, info.lat), crs="EPSG:4326")

#Project to GDA2020 Albers (EPSG 9473), national standard for Australia and covers whole area
gdf_utm = gdf.to_crs(epsg=9473)

# Extract x,y in meters
X = np.vstack([gdf_utm.geometry.x, gdf_utm.geometry.y]).T

# Step 3: Run KMeans
inertias = []
K_range = range(1, 20)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

# Step 4: Elbow plot
plt.plot(K_range, inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.title("WCSS for k-means clustering")
plt.show()

In [ ]:
#Get cluster assignments
best_k = 6
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
info["cluster"] = kmeans.fit_predict(X)


In [ ]:
cluster_styles = {
    0: {"name": "Alpine NSW", "color": "#e78ac3"},
    1: {"name": "South Australia", "color": "#8da0cb"},
    2: {"name": "North Queensland", "color": "#fc8d62"},
    3: {"name": "West Victoria", "color": "#ffd92f"},
    4: {"name": "Eastern Central", "color": "#a6d854"},
    5: {"name": "Bass Strait", "color": "#66c2a5"},
}

id_to_name = {k: v["name"] for k, v in cluster_styles.items()}
id_to_color = {k: v["color"] for k, v in cluster_styles.items()}
name_to_color = {v["name"]: v["color"] for v in cluster_styles.values()}

info["cluster_name"] = info["cluster"].map(id_to_name)
info["cluster_color"] = info["cluster"].map(id_to_color)

In [ ]:
# Convert centroids back to lat/lon for plotting
centers_utm = kmeans.cluster_centers_
centers_gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(centers_utm[:,0], centers_utm[:,1]),
    crs="EPSG:9473"
)
centers_latlon = centers_gdf.to_crs(epsg=4326)
centers_df = pd.DataFrame({
    "lat": centers_latlon.geometry.y,
    "lon": centers_latlon.geometry.x,
    "cluster": [f"Centroid {i}" for i in range(len(centers_utm))]
})

info['cluster_str'] = info['cluster'].astype(str)

fig = px.scatter_map(
    info,
    lat="lat_jittered",
    lon="lon_jittered",
    hover_name="DUID",
    hover_data={"lat": False, "lon": False, "cluster": True, "cluster_name": True},
    color="cluster_name",
    color_discrete_map=name_to_color,  # fixed colors per named cluster
    zoom=3,
    map_style="carto-positron"
)

# Add centroids with bright color and bigger size
fig.add_scattermap(
    lat=centers_df["lat"],
    lon=centers_df["lon"],
    mode="markers",
    marker=dict(size=7, color='white', symbol='star'), 
    text=centers_df["cluster"],
    hoverinfo="text",
    name="Cluster Centroids"
)

fig.update_layout(
    title="K-means clusters of NEM wind farms",
    legend_title_text="Cluster ID (0-5)"
)

fig.update_layout(title="K-means clusters of NEM wind farms")
fig.show()

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([130, 155, -45, -10], crs=ccrs.PlateCarree())
ax.coastlines(resolution='10m', color='black')

ax.add_feature(cfeature.STATES, linestyle='-', lw=0.5)

# Plot your points and keep the scatter object
sc = ax.scatter(
    info["lon_jittered"], info["lat_jittered"],
    c=info["cluster_color"],         # use fixed hex colors
    s=200, alpha=0.7,
    transform=ccrs.PlateCarree()
)
ax.scatter(
    centers_df["lon"], centers_df["lat"],
    marker='*', s=200, c='black', edgecolor='black',
    label='Centroids', zorder=5, transform=ccrs.PlateCarree()
)

# Get unique clusters for legend
cluster_labels = sorted(np.unique(info["cluster"]))

cluster_handles = [
    mpatches.Patch(
        color=cluster_styles[c]["color"],
        label=cluster_styles[c]["name"]
    )
    for c in cluster_labels
]

centroid_handle = mlines.Line2D([], [], color='black', marker='*', linestyle='None',
                                markeredgecolor='black', markersize=20, label='Centroid')

ax.legend(handles=cluster_handles + [centroid_handle], loc='upper left', title="Legend", fontsize=14)

plt.title("K-means clusters of NEM wind farms", fontsize=20)
plt.savefig("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/wind_chapter/clusters_map.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df_clust = pd.merge(df,info[["DUID","cluster"]], on='DUID', how='left')
day_counts = pd.crosstab(df_clust['cluster'], df_clust['EHF_flag']).rename(columns={0: 'baseline days', 1: 'heatwave days'}).reset_index()

In [ ]:
df_clust

In [ ]:
# unique IDs per cluster
id_counts = df_clust.groupby('cluster')['DUID'].nunique().reset_index(name='unit count')

# merge into the crosstab result
day_counts = day_counts.merge(id_counts, on='cluster')
day_counts

In [ ]:
# prep
df_clust['time'] = pd.to_datetime(df_clust['time'])

# per-ID first and last active day + cluster
id_meta = (
    df_clust.groupby('DUID', as_index=False)
      .agg(first_day=('time', 'min'),
           last_day=('time', 'max'))
    .merge(df_clust[['DUID','cluster']].drop_duplicates(), on='DUID')
)
# order IDs by first_day for nicer plotting
id_meta = id_meta.sort_values(['cluster', 'first_day'])

# count first-online events per day+cluster, reindex to full date range and cumsum
first_counts = (
    id_meta.groupby(['first_day', 'cluster'])['DUID']
           .count()
           .rename('new_ids')
           .reset_index()
)

full_dates = pd.date_range(df_clust['time'].min(), df_clust['time'].max(), freq='D')
first_pivot = first_counts.pivot(index='first_day', columns='cluster', values='new_ids').reindex(full_dates, fill_value=0)
cum_by_cluster = first_pivot.cumsum()

# plot stepped lines
fig, ax = plt.subplots(figsize=(12, 6))

for col in cum_by_cluster.columns:
    style = cluster_styles.get(col, {})
    ax.step(
        cum_by_cluster.index,
        cum_by_cluster[col],
        where='post',
        color=style.get("color", None),
        label=style.get("name", col)
    )

ax.set_xlabel('Day')
ax.set_ylabel('Cumulative number of IDs online')
ax.set_title('Cumulative units online by cluster (step plot)')
ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
def _haversine_matrix(lons, lats):
    """Pairwise haversine distances (m) between 1D arrays of lon/lat in degrees."""
    R = 6371000.0  # Earth radius (m)
    lon_rad = np.deg2rad(lons)
    lat_rad = np.deg2rad(lats)
    dlat = lat_rad[:, None] - lat_rad[None, :]
    dlon = lon_rad[:, None] - lon_rad[None, :]
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat_rad[:, None]) * np.cos(lat_rad[None, :]) * np.sin(dlon / 2.0) ** 2
    a = np.clip(a, 0.0, 1.0)
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def _meters_to_deg_scales(latitudes):
    """
    Return (lon_deg_per_meter, lat_deg_per_meter) for each latitude.
    latitudes: array in degrees
    """
    lat_deg_m = 111320.0
    lat_rad = np.deg2rad(latitudes)
    lon_deg_m = lat_deg_m * np.cos(lat_rad)
    lon_scale = 1.0 / lon_deg_m
    lat_scale = 1.0 / lat_deg_m
    return lon_scale, lat_scale

def plot_single_cluster(
    info_df,
    cluster_id,
    padding=1.5,
    land_color="lightgray",
    ocean_color="lightblue",
    tiles=False,
    tile_provider="terrain-background",
    tile_zoom=6,
    figsize=(10, 10),
    marker_size=60,
    alpha=1,
    cmap_name="tab20",
    max_legend_items=200,
    # Jitter / separation options
    jitter=True,
    jitter_mode="meters",      # "deg" or "meters"
    jitter_deg=0.02,           # used when jitter_mode == "deg" (std dev in degrees)
    jitter_meters=5000.0,      # used when jitter_mode == "meters" (std dev in meters)
    jitter_scale=1.0,          # scale multiplier for jitter magnitude
    jitter_seed=None,
    enforce_min_separation=True,
    min_separation_m=2000.0,   # minimum allowed separation (meters) after jitter
    max_iter=50
):
    """
    Plot a single cluster, colour each unit (DUID) uniquely and place the legend
    to the right outside the axes. Points in the selected cluster are jittered
    and (optionally) iteratively separated until they meet min_separation_m.

    Args:
        info_df (pd.DataFrame): DataFrame with columns ['lon','lat','cluster','DUID'].
        cluster_id (int): cluster to plot.
        jitter_mode: "deg" (jitter in degrees) or "meters" (jitter in meters converted to degrees).
        jitter_deg, jitter_meters, jitter_scale: control jitter magnitude.
        enforce_min_separation: if True, iteratively push points apart until min_separation_m.
        min_separation_m: desired minimum separation in meters.
        max_iter: max iterations for separation loop (safety).
    Returns:
        fig, ax
    """
    cluster_data = info_df[info_df["cluster"] == cluster_id].copy()
    if cluster_data.empty:
        print(f"No data found for cluster {cluster_id}")
        return None, None

    # Base coordinates (do not assume lon_jittered present)
    lons = cluster_data["lon"].values.astype(float)
    lats = cluster_data["lat"].values.astype(float)
    n = len(lons)

    rng = np.random.RandomState(jitter_seed) if jitter_seed is not None else np.random

    # Compute initial jitter offsets
    if jitter and n > 0:
        if jitter_mode == "deg":
            std_deg = float(jitter_deg) * float(jitter_scale)
            lon_offsets = rng.normal(loc=0.0, scale=std_deg, size=n)
            lat_offsets = rng.normal(loc=0.0, scale=std_deg, size=n)
        elif jitter_mode == "meters":
            std_m = float(jitter_meters) * float(jitter_scale)
            lon_scale_per_m, lat_scale_per_m = _meters_to_deg_scales(lats)
            # lon_scale_per_m and lat_scale_per_m are arrays per-point (degrees per meter)
            # convert std_m to per-point degree std
            lon_std_deg = lon_scale_per_m * std_m
            lat_std_deg = lat_scale_per_m * std_m
            # sample per-point normals
            lon_offsets = rng.normal(loc=0.0, scale=lon_std_deg, size=n)
            lat_offsets = rng.normal(loc=0.0, scale=lat_std_deg, size=n)
        else:
            raise ValueError("jitter_mode must be 'deg' or 'meters'")
    else:
        lon_offsets = np.zeros(n)
        lat_offsets = np.zeros(n)

    lon_j = lons + lon_offsets
    lat_j = lats + lat_offsets

    # Iteratively enforce minimum separation (pairwise repulsion)
    if enforce_min_separation and n > 1:
        if n > 2000:
            warnings.warn("enforce_min_separation uses O(N^2) pairwise checks and may be slow for large N.")
        it = 0
        while it < max_iter:
            it += 1
            dmat = _haversine_matrix(lon_j, lat_j)
            np.fill_diagonal(dmat, np.inf)
            min_d = np.min(dmat)
            if min_d >= min_separation_m:
                break
            # find violating pairs (only those < min_separation_m)
            viol_i, viol_j = np.where(dmat < min_separation_m)
            if len(viol_i) == 0:
                break
            # accumulate displacement in degrees
            dx = np.zeros(n)
            dy = np.zeros(n)
            # For each violating pair push them apart a small amount proportional to overlap
            for i1, i2 in zip(viol_i, viol_j):
                # compute vector from i2 -> i1 in degrees
                dlon = lon_j[i1] - lon_j[i2]
                dlat = lat_j[i1] - lat_j[i2]
                # break exact ties with tiny random perturbation
                if dlon == 0 and dlat == 0:
                    dlon = (rng.rand() - 0.5) * 1e-6
                    dlat = (rng.rand() - 0.5) * 1e-6
                # magnitude in degree space (approx)
                deg_dist = np.hypot(dlon, dlat)
                if deg_dist == 0:
                    continue
                # overlap fraction (how much below the minimum)
                over = (min_separation_m - dmat[i1, i2]) / max(min_separation_m, 1.0)
                # small push factor to avoid oscillation; tuneable
                push = 0.3 * over
                dx_i = (dlon / deg_dist) * push
                dy_i = (dlat / deg_dist) * push
                dx[i1] += dx_i
                dy[i1] += dy_i
                dx[i2] -= dx_i
                dy[i2] -= dy_i
            lon_j = lon_j + dx
            lat_j = lat_j + dy
        else:
            warnings.warn(
                f"Reached max_iter={max_iter} while enforcing separation; final min distance = "
                f"{np.min(_haversine_matrix(lon_j, lat_j)):.1f} m"
            )

    # Attach jittered coords to a local copy (no mutation of original info_df)
    cluster_data = cluster_data.reset_index(drop=True)
    cluster_data["lon_jittered"] = lon_j
    cluster_data["lat_jittered"] = lat_j

    # compute dynamic extent from jittered coords
    lon_min = cluster_data["lon_jittered"].min() - padding
    lon_max = cluster_data["lon_jittered"].max() + padding
    lat_min = cluster_data["lat_jittered"].min() - padding
    lat_max = cluster_data["lat_jittered"].max() + padding
    extent = [lon_min, lon_max, lat_min, lat_max]

    # Determine unique DUIDs and create colors
    unique_duids = list(cluster_data["DUID"].unique())
    n_duids = len(unique_duids)
    if (max_legend_items is not None) and (n_duids > max_legend_items):
        print(
            f"WARNING: cluster {cluster_id} has {n_duids} units; truncating legend to first "
            f"{max_legend_items} entries. Set max_legend_items=None to show all."
        )
        legend_duids = unique_duids[:max_legend_items]
    else:
        legend_duids = unique_duids

    try:
        cmap = plt.get_cmap(cmap_name, n_duids)
        cmap_colors = cmap(np.arange(n_duids))
    except Exception:
        base = plt.get_cmap("tab20")
        cmap_colors = base(np.arange(n_duids) % base.N)

    duid_to_color = {duid: cmap_colors[i] for i, duid in enumerate(unique_duids)}
    point_colors = [duid_to_color[d] for d in cluster_data["DUID"].values]

    # Create figure and axis, set extent
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    # Background
    if tiles:
        tiler = cimgt.Stamen(tile_provider)
        ax.add_image(tiler, tile_zoom, resample=True)
    else:
        ax.add_feature(cfeature.OCEAN, facecolor=ocean_color, zorder=0)
        ax.add_feature(cfeature.LAND, facecolor=land_color, zorder=1)
        ax.add_feature(cfeature.LAKES, facecolor=ocean_color, zorder=1)
        ax.add_feature(cfeature.RIVERS, edgecolor="blue", zorder=2, linewidth=0.5)

    ax.coastlines(resolution="10m", color="black", linewidth=0.6, zorder=3)
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False

    # Scatter jittered points
    sc = ax.scatter(
        cluster_data["lon_jittered"].values,
        cluster_data["lat_jittered"].values,
        facecolors=point_colors,      # use facecolors explicitly
        edgecolors='black',           # black outline
        linewidths=0.8,               # outline width in points
        s=marker_size,
        alpha=alpha,
        transform=ccrs.PlateCarree(),
        zorder=4
    )

    # Legend patches (possibly truncated)
    legend_handles = [
        mpatches.Patch(color=duid_to_color[duid], label=str(duid))
        for duid in legend_duids
    ]

    # Place legend to the right outside axes
    plt.subplots_adjust(right=0.72)
    ax.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        title="DUID",
        frameon=True,
        fontsize=12,
        title_fontsize=16
    )

    name = cluster_data["cluster_name"][0]
    plt.title(f"Wind farms in {name} Cluster", fontsize=16)
    plt.show()
    return fig, ax

import os
os.makedirs("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/wind_chapter/cluster_maps", exist_ok=True)
plt.ioff()  # avoid pop-up windows

for cid in sorted(info["cluster"].unique()):
    fig, ax = plot_single_cluster(info, cid, enforce_min_separation=True, padding=1.5, jitter=True)
    if fig is None:
        continue
    out = f"/g/data/ng72/ms5578/ID_HW_BARRA/data/output/wind_chapter/cluster_maps/cluster_{cid}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)

plt.ion()  # re-enable interactive plotting if desired

In [ ]:
# Compute silhouette scores for each point
sil_scores = silhouette_samples(X, info['cluster'])

# Add to your df
info['silhouette'] = sil_scores

# Average silhouette per cluster
for c in range(best_k):
    avg_score = info[info['cluster']==c]['silhouette'].mean()
    print(f"Cluster {c}: avg silhouette = {avg_score:.3f}")

# Get unique cluster labels in the same order as your scatter plot
cluster_labels = np.unique(info["cluster"])

# Use the same colormap and normalization as your scatter plot
cmap = plt.get_cmap('tab10')
norm = plt.Normalize(cluster_labels.min(), cluster_labels.max())

def plot_silhouette_hist(info, cluster_labels):
    fig, ax = plt.subplots(figsize=(8,5))
    for c in cluster_labels:
        cluster_scores = info[info['cluster'] == c]['silhouette']
        color = cluster_styles[c]["color"]   # fixed per cluster
        label = cluster_styles[c]["name"]
        ax.hist(cluster_scores, bins=10, alpha=0.6, label=label, color=color)
    ax.set_xlabel('Silhouette score', fontsize=12)
    ax.set_ylabel('Number of points', fontsize=12)
    ax.set_title('Silhouette Distribution per Cluster', fontsize=18)
    ax.legend(fontsize=12)
    return fig

# Call the function
fig = plot_silhouette_hist(info, cluster_labels)
fig.savefig("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/wind_chapter/silhouette.png",
            dpi=300, bbox_inches="tight")

In [ ]:
clustered_info = info[['station_name','DUID', 'lat', 'lon', 'cluster', 'cluster_name', 'cluster_color']]
clustered_info.to_csv("data/preprocess/wind_spatial_clusters.csv", index=False)